In [0]:
%pip install pydicom simplejson s3fs --quiet
dbutils.library.restartPython()

In [0]:
import hashlib
import json
import os
import time
from dataclasses import dataclass
from typing import Iterator, Tuple

from pyspark.sql.datasource import (
    DataSource,
    DataSourceReader,
    DataSourceStreamReader,
    InputPartition,
)
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, BooleanType, DoubleType, IntegerType


# ---------------------------------------------------------------------------
# Schema definition: combines file catalog columns + core DICOM metadata
# ---------------------------------------------------------------------------
DICOM_SCHEMA = StructType([
    # File-level columns (like binaryFile / cloudFiles)
    StructField("path", StringType(), False),
    StructField("modificationTime", StringType(), True),
    StructField("file_size", LongType(), True),
    # Core DICOM identifiers
    StructField("SOPClassUID", StringType(), True),
    StructField("SOPInstanceUID", StringType(), True),
    StructField("StudyInstanceUID", StringType(), True),
    StructField("SeriesInstanceUID", StringType(), True),
    StructField("PatientID", StringType(), True),
    StructField("PatientName", StringType(), True),
    StructField("StudyDate", StringType(), True),
    StructField("StudyTime", StringType(), True),
    StructField("StudyDescription", StringType(), True),
    StructField("SeriesDescription", StringType(), True),
    StructField("Modality", StringType(), True),
    StructField("Manufacturer", StringType(), True),
    StructField("InstitutionName", StringType(), True),
    StructField("InstanceNumber", StringType(), True),
    StructField("SeriesNumber", StringType(), True),
    StructField("SliceThickness", StringType(), True),
    StructField("PixelSpacing", StringType(), True),
    StructField("Rows", IntegerType(), True),
    StructField("Columns", IntegerType(), True),
    StructField("BitsAllocated", IntegerType(), True),
    StructField("ImageOrientationPatient", StringType(), True),
    StructField("ImagePositionPatient", StringType(), True),
    StructField("TransferSyntaxUID", StringType(), True),
    # Pixel stats (when deep=True)
    StructField("has_pixel", BooleanType(), True),
    StructField("img_min", DoubleType(), True),
    StructField("img_max", DoubleType(), True),
    StructField("img_avg", DoubleType(), True),
    # Full metadata JSON (VARIANT-ready)
    StructField("meta", StringType(), True),
    # Error handling
    StructField("_error", StringType(), True),
])


# ---------------------------------------------------------------------------
# Partition class: each partition is a batch of file paths
# ---------------------------------------------------------------------------
@dataclass
class DicomFilePartition(InputPartition):
    files: list  # list of (path, mod_time, file_size) tuples


# ---------------------------------------------------------------------------
# Helper: extract metadata from a single DICOM file
# ---------------------------------------------------------------------------
def _extract_dicom_metadata(path: str, deep: bool = False) -> dict:
    """Read a DICOM file and extract metadata. Returns a flat dict matching DICOM_SCHEMA."""
    import simplejson
    from pydicom import dcmread

    result = {"path": path, "_error": None}

    try:
        # Open file - supports local paths and /Volumes/ paths
        if path.startswith("s3://"):
            import s3fs
            fs = s3fs.S3FileSystem(anon=False)
            fp = fs.open(path)
            fsize = fs.size(path)
        else:
            fp = open(path, "rb")
            fsize = os.stat(path).st_size

        result["file_size"] = fsize

        with dcmread(fp, defer_size=1000, stop_before_pixels=(not deep)) as ds:
            # Extract core identifiers using DICOM keyword access
            for keyword in [
                "SOPClassUID", "SOPInstanceUID", "StudyInstanceUID",
                "SeriesInstanceUID", "PatientID", "PatientName",
                "StudyDate", "StudyTime", "StudyDescription",
                "SeriesDescription", "Modality", "Manufacturer",
                "InstitutionName", "InstanceNumber", "SeriesNumber",
                "SliceThickness", "PixelSpacing",
                "ImageOrientationPatient", "ImagePositionPatient",
            ]:
                val = getattr(ds, keyword, None)
                if val is not None:
                    result[keyword] = str(val)
                else:
                    result[keyword] = None

            # Integer fields
            for keyword in ["Rows", "Columns", "BitsAllocated"]:
                val = getattr(ds, keyword, None)
                result[keyword] = int(val) if val is not None else None

            # TransferSyntaxUID from file_meta
            if hasattr(ds, "file_meta") and ds.file_meta is not None:
                tsuid = getattr(ds.file_meta, "TransferSyntaxUID", None)
                result["TransferSyntaxUID"] = str(tsuid) if tsuid else None
            else:
                result["TransferSyntaxUID"] = None

            # Pixel statistics (deep mode)
            if deep:
                try:
                    import numpy as np
                    pixel_array = ds.pixel_array
                    result["has_pixel"] = True
                    result["img_min"] = float(np.min(pixel_array))
                    result["img_max"] = float(np.max(pixel_array))
                    result["img_avg"] = float(np.average(pixel_array))
                except Exception:
                    result["has_pixel"] = False
                    result["img_min"] = None
                    result["img_max"] = None
                    result["img_avg"] = None
            else:
                result["has_pixel"] = None
                result["img_min"] = None
                result["img_max"] = None
                result["img_avg"] = None

            # Full metadata JSON
            try:
                # Remove pixel data before serialization
                if "7FE00010" in ds:
                    del ds["7FE00010"]
                if "60003000" in ds:
                    del ds["60003000"]
                js = ds.to_json_dict()
                if hasattr(ds, "file_meta") and ds.file_meta is not None:
                    js.update(ds.file_meta.to_json_dict())
                result["meta"] = simplejson.dumps(js, ignore_nan=True)
            except Exception:
                result["meta"] = None

    except Exception as e:
        result["_error"] = str(e)
        # Fill remaining fields with None
        for field in DICOM_SCHEMA.fieldNames():
            if field not in result:
                result[field] = None

    return result


print("✓ DICOM DataSource core components defined")

In [0]:
# ---------------------------------------------------------------------------
# Batch Reader: scans a directory for DICOM files and extracts metadata
# ---------------------------------------------------------------------------
class DicomBatchReader(DataSourceReader):
    """Reads DICOM files from a directory, extracts metadata in parallel partitions."""

    def __init__(self, schema, options):
        self.schema = schema
        self.path = options.get("path", options.get("basePath", ""))
        self.pattern = options.get("pathGlobFilter", "*.dcm")
        self.recurse = options.get("recursiveFileLookup", "true").lower() == "true"
        self.deep = options.get("deep", "false").lower() == "true"
        self.partition_size = int(options.get("filesPerPartition", "50"))

    def partitions(self):
        """Discover DICOM files and split them into partitions."""
        import glob
        import fnmatch

        all_files = []
        base = self.path

        if self.recurse:
            for root, dirs, files in os.walk(base):
                for f in files:
                    if fnmatch.fnmatch(f, self.pattern) or f.lower().endswith(".dcm"):
                        full_path = os.path.join(root, f)
                        try:
                            stat = os.stat(full_path)
                            all_files.append((full_path, str(stat.st_mtime), stat.st_size))
                        except OSError:
                            all_files.append((full_path, None, None))
        else:
            for f in os.listdir(base):
                if fnmatch.fnmatch(f, self.pattern) or f.lower().endswith(".dcm"):
                    full_path = os.path.join(base, f)
                    try:
                        stat = os.stat(full_path)
                        all_files.append((full_path, str(stat.st_mtime), stat.st_size))
                    except OSError:
                        all_files.append((full_path, None, None))

        # Also include files without extension (common in DICOM)
        if self.pattern == "*.dcm":
            # Re-scan for extensionless files (DICOM often has no extension)
            pass  # the fnmatch above handles *.dcm; extensionless handled by .dcm check

        # Split into partitions
        partitions = []
        for i in range(0, len(all_files), self.partition_size):
            batch = all_files[i:i + self.partition_size]
            partitions.append(DicomFilePartition(files=batch))

        if not partitions:
            partitions.append(DicomFilePartition(files=[]))

        return partitions

    def read(self, partition: DicomFilePartition) -> Iterator[Tuple]:
        """Read a partition of DICOM files and yield metadata tuples."""
        from concurrent.futures import ThreadPoolExecutor

        deep = self.deep
        field_names = [f.name for f in DICOM_SCHEMA.fields]

        def _process(file_info):
            path, mod_time, fsize = file_info
            result = _extract_dicom_metadata(path, deep=deep)
            result["modificationTime"] = mod_time
            if fsize is not None and result.get("file_size") is None:
                result["file_size"] = fsize
            return tuple(result.get(fn) for fn in field_names)

        # Use thread pool for concurrent I/O (same pattern as DicomMetaExtractor)
        with ThreadPoolExecutor(max_workers=min(32, len(partition.files) or 1)) as executor:
            results = executor.map(_process, partition.files)
            for row in results:
                yield row


print("✓ DicomBatchReader defined")

In [0]:
# ---------------------------------------------------------------------------
# Stream Reader: incrementally discovers new DICOM files and extracts metadata
# ---------------------------------------------------------------------------
class DicomStreamReader(DataSourceStreamReader):
    """
    Streaming reader that watches a directory for new DICOM files.
    Each microbatch processes newly discovered files since the last offset.
    The offset is a JSON dict tracking the set of already-processed file paths.
    """

    def __init__(self, schema, options):
        self.schema = schema
        self.path = options.get("path", options.get("basePath", ""))
        self.pattern = options.get("pathGlobFilter", "*.dcm")
        self.recurse = options.get("recursiveFileLookup", "true").lower() == "true"
        self.deep = options.get("deep", "false").lower() == "true"
        self.max_files_per_trigger = int(options.get("maxFilesPerTrigger", "1000"))
        self.partition_size = int(options.get("filesPerPartition", "50"))
        self._known_files = set()  # Track all discovered files
        self._current_new_files = []  # Files discovered in latest scan

    def _scan_directory(self) -> list:
        """Scan the directory and return list of (path, mod_time, size) tuples."""
        import fnmatch

        all_files = []
        base = self.path

        def _is_dicom_candidate(filename):
            """Check if file could be a DICOM file (extension or extensionless)."""
            lower = filename.lower()
            if fnmatch.fnmatch(lower, self.pattern.lower()):
                return True
            if lower.endswith(".dcm") or lower.endswith(".dicom"):
                return True
            # Include extensionless files (common in DICOM)
            if "." not in filename:
                return True
            return False

        if self.recurse:
            for root, dirs, files in os.walk(base):
                for f in files:
                    if _is_dicom_candidate(f):
                        full_path = os.path.join(root, f)
                        try:
                            stat = os.stat(full_path)
                            all_files.append((full_path, str(stat.st_mtime), stat.st_size))
                        except OSError:
                            all_files.append((full_path, None, None))
        else:
            if os.path.isdir(base):
                for f in os.listdir(base):
                    if _is_dicom_candidate(f):
                        full_path = os.path.join(base, f)
                        try:
                            stat = os.stat(full_path)
                            all_files.append((full_path, str(stat.st_mtime), stat.st_size))
                        except OSError:
                            all_files.append((full_path, None, None))

        return all_files

    def initialOffset(self) -> dict:
        """Returns the initial offset — empty set means no files processed yet."""
        return {"processed_count": 0, "processed_files": []}

    def latestOffset(self) -> dict:
        """Scan directory, identify new files, return new offset."""
        all_files = self._scan_directory()
        all_paths = {f[0] for f in all_files}

        # Find new files not yet processed
        new_paths = all_paths - self._known_files
        new_files = [(p, mt, sz) for p, mt, sz in all_files if p in new_paths]

        # Respect maxFilesPerTrigger
        new_files = new_files[:self.max_files_per_trigger]
        self._current_new_files = new_files

        # Update known files
        new_processed = self._known_files | {f[0] for f in new_files}

        return {
            "processed_count": len(new_processed),
            "processed_files": sorted(list(new_processed))[-100:]  # Keep last 100 for offset size management
        }

    def partitions(self, start: dict, end: dict) -> list:
        """Split newly discovered files into partitions for parallel processing."""
        files_to_process = self._current_new_files

        if not files_to_process:
            return [DicomFilePartition(files=[])]

        partitions = []
        for i in range(0, len(files_to_process), self.partition_size):
            batch = files_to_process[i:i + self.partition_size]
            partitions.append(DicomFilePartition(files=batch))

        return partitions

    def read(self, partition: DicomFilePartition) -> Iterator[Tuple]:
        """Read a partition of DICOM files and yield metadata tuples."""
        from concurrent.futures import ThreadPoolExecutor

        deep = self.deep
        field_names = [f.name for f in DICOM_SCHEMA.fields]

        def _process(file_info):
            path, mod_time, fsize = file_info
            result = _extract_dicom_metadata(path, deep=deep)
            result["modificationTime"] = mod_time
            if fsize is not None and result.get("file_size") is None:
                result["file_size"] = fsize
            return tuple(result.get(fn) for fn in field_names)

        if not partition.files:
            return

        with ThreadPoolExecutor(max_workers=min(32, len(partition.files))) as executor:
            results = executor.map(_process, partition.files)
            for row in results:
                yield row

    def commit(self, end: dict):
        """Mark files as processed."""
        self._known_files = set(end.get("processed_files", []))
        # Also add by count tracking
        for f in self._current_new_files:
            self._known_files.add(f[0])

    def stop(self):
        """Cleanup on stream stop."""
        pass


print("✓ DicomStreamReader defined")

In [0]:
# ---------------------------------------------------------------------------
# DataSource class: registers 'dicom' format with Spark
# ---------------------------------------------------------------------------
class DicomDataSource(DataSource):
    """
    PySpark custom DataSource for DICOM files.

    Integrates file discovery with metadata extraction for better performance:
    - Eliminates the separate DicomMetaExtractor Transformer step
    - Supports both batch and streaming reads
    - Uses concurrent I/O via ThreadPoolExecutor (same as DicomMetaExtractor)
    - Outputs structured schema with core DICOM fields + full JSON metadata

    Options:
        path (str): Base directory containing DICOM files
        pathGlobFilter (str): File pattern filter (default: "*.dcm")
        recursiveFileLookup (str): "true"/"false" (default: "true")
        deep (str): "true"/"false" - extract pixel statistics (default: "false")
        maxFilesPerTrigger (str): Max files per streaming microbatch (default: "1000")
        filesPerPartition (str): Files per reader partition (default: "50")

    Usage (batch):
        df = spark.read.format("dicom").load("/path/to/dicoms")

    Usage (streaming):
        df = spark.readStream.format("dicom").load("/path/to/dicoms")
    """

    @classmethod
    def name(cls):
        return "dicom"

    def schema(self):
        return DICOM_SCHEMA

    def reader(self, schema) -> DicomBatchReader:
        return DicomBatchReader(schema, self.options)

    def streamReader(self, schema) -> DicomStreamReader:
        return DicomStreamReader(schema, self.options)


# Register the data source with Spark
spark.dataSource.register(DicomDataSource)
print("✓ DICOM DataSource registered as format 'dicom'")
print("  Usage: spark.read.format('dicom').load('/path/to/dicoms')")
print("  Usage: spark.readStream.format('dicom').load('/path/to/dicoms')")

In [0]:
# ---------------------------------------------------------------------------
# Batch read: read all DICOM files from the test volume
# ---------------------------------------------------------------------------
DICOM_PATH = "/Volumes/serverless_pixels_release_catalog/tcia/pixels_volume/unzipped/"

df = (
    spark.read
    .format("dicom")
    .option("pathGlobFilter", "*")  # Include extensionless DICOM files
    .option("recursiveFileLookup", "true")
    .option("deep", "false")
    .option("filesPerPartition", "32")
    .load(DICOM_PATH)
    .limit(100)
)

#print(f"Discovered {df.count()} DICOM files")
display(df.select("path", "PatientID", "StudyInstanceUID", "SeriesInstanceUID", "Modality", "StudyDescription", "Rows", "Columns", "file_size", "_error"))

In [0]:
# ---------------------------------------------------------------------------
# Streaming ingestion: incrementally process new DICOM files into a Delta table
# ---------------------------------------------------------------------------
from pyspark.sql.functions import col, current_timestamp, parse_json

CATALOG = "serverless_pixels_release_catalog"
SCHEMA = "pixels_samples"
TABLE = f"{CATALOG}.{SCHEMA}.dicom_metadata_stream"
CHECKPOINT = f"/Volumes/{CATALOG}/{SCHEMA}/pixels_volume/_checkpoints/dicom_datasource"

# Create the streaming DataFrame
stream_df = (
    spark.readStream
    .format("dicom")
    .option("pathGlobFilter", "*")
    .option("recursiveFileLookup", "true")
    .option("deep", "false")
    .option("maxFilesPerTrigger", "500")
    .option("filesPerPartition", "32")
    .load(DICOM_PATH)
)

# Add ingestion timestamp and convert meta to VARIANT
stream_df = (
    stream_df
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("meta_variant", parse_json(col("meta")))
    .drop("meta")
    .withColumnRenamed("meta_variant", "meta")
)

# Write stream to Delta table
query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)  # Process all available files then stop
    .toTable(TABLE)
)

query.awaitTermination()
print(f"✓ Stream completed. Data written to {TABLE}")

In [0]:
# ---------------------------------------------------------------------------
# Verify: query the Delta table written by the stream
# ---------------------------------------------------------------------------
result_df = spark.read.table(TABLE)
print(f"Total rows in {TABLE}: {result_df.count()}")
print(f"Unique studies: {result_df.select('StudyInstanceUID').distinct().count()}")
print(f"Unique series: {result_df.select('SeriesInstanceUID').distinct().count()}")
print(f"Modalities: {[r.Modality for r in result_df.select('Modality').distinct().collect()]}")
print()
display(
    result_df.select(
        "PatientID", "StudyInstanceUID", "SeriesInstanceUID", 
        "Modality", "StudyDescription", "Rows", "Columns",
        "file_size", "ingestion_time", "_error"
    ).limit(20)
)